# Lesson 13B: Diffusion Models - Practical

Train a simple diffusion model to generate 2D data.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
np.random.seed(42)
plt.style.use('seaborn-v0_8-darkgrid')
print("✅ Loaded!")

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.datasets import make_moons

class DiffusionModel(nn.Module):
    def __init__(self, input_dim=2, time_dim=32, hidden_dim=64):
        super().__init__()
        self.time_embed = nn.Sequential(
            nn.Linear(1, time_dim),
            nn.ReLU(),
        )
        self.net = nn.Sequential(
            nn.Linear(input_dim + time_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, input_dim)
        )
    
    def forward(self, x, t):
        t_embed = self.time_embed(t.unsqueeze(-1))
        x_t = torch.cat([x, t_embed], dim=-1)
        return self.net(x_t)

def get_beta_schedule(T=100):
    return torch.linspace(0.0001, 0.02, T)

def forward_diffusion(x0, t, betas):
    alpha = 1 - betas
    alpha_bar = torch.cumprod(alpha, dim=0)
    alpha_t = alpha_bar[t]
    noise = torch.randn_like(x0)
    x_t = torch.sqrt(alpha_t) * x0 + torch.sqrt(1 - alpha_t) * noise
    return x_t, noise

# Prepare data
X, _ = make_moons(n_samples=1000, noise=0.1, random_state=42)
X = torch.FloatTensor(X)

# Setup
T = 100
betas = get_beta_schedule(T)
model = DiffusionModel(input_dim=2)
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Train
print("Training diffusion model...")
for epoch in range(200):
    # Sample batch and random timesteps
    batch = X[torch.randint(0, len(X), (64,))]
    t = torch.randint(0, T, (64,))
    
    # Forward diffusion
    x_noisy, noise = forward_diffusion(batch, t, betas)
    
    # Predict noise
    t_float = t.float()
    noise_pred = model(x_noisy, t_float)
    
    # Loss
    loss = nn.MSELoss()(noise_pred, noise)
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    if (epoch + 1) % 50 == 0:
        print(f"Epoch {epoch+1}/200, Loss: {loss.item():.4f}")

# Sample (simplified reverse process)
@torch.no_grad()
def sample(model, n_samples=500, T=100):
    x = torch.randn(n_samples, 2)
    for t in reversed(range(T)):
        t_batch = torch.full((n_samples,), t, dtype=torch.float32)
        noise_pred = model(x, t_batch)
        # Simplified denoising step
        x = x - 0.01 * noise_pred
    return x

generated = sample(model, n_samples=500, T=T)

# Visualize
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.scatter(X[:, 0], X[:, 1], alpha=0.5, s=10)
plt.title('Real Data', fontweight='bold')
plt.subplot(1, 2, 2)
plt.scatter(generated[:, 0], generated[:, 1], alpha=0.5, s=10, c='red')
plt.title('Diffusion Model Generated', fontweight='bold')
plt.tight_layout()
plt.show()
print("✅ Diffusion model trained and generated samples!")